<a href="https://colab.research.google.com/github/ez4tien/training_ai_arena/blob/main/SoVITS-forked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS WebUI

## Env Setup (Run Once Only)


### 1.

In [1]:
%%writefile /content/setup.sh
set -e

cd /content

git clone https://github.com/RVC-Boss/GPT-SoVITS.git

cd GPT-SoVITS

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS

pip install ipykernel

bash install.sh --device CU126 --source HF --download-uvr5

Writing /content/setup.sh


### 2.

In [2]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
!cd /content && bash setup.sh


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:01:06
🔁 Restarting kernel...
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 6201, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 6201 (delta 71), reused 51 (delta 50), pack-reused 6088 (from 2)
Receiving objects: 100% (6201/6201), 14.66 MiB | 19.32 MiB/s, 

In [3]:
!/usr/local/envs/GPTSoVITS/bin/pip install "fastapi<0.113.0" "starlette<0.38.0" "pydantic<2.10.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 114.6 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Uninstalling pydantic_core-2.27.2:
      Successfully uninstalled pydantic_core-2.27.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.10.6
    Uninstalling pydantic-2.10.6:
      Successfully uninstalled pydantic-2.10.6
  Attempting uninstall: starlette
    Found existing installation: starlette 1.7.0
    Uninstalling starlette-1.7.0:
      Successfully uninstalled starlette-1.7.0
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.162.0
    Uninstalling fastapi-0.162.0:
      Successfully uninstalled fastapi-0.162.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastapi]


In [1]:
!rm /usr/local/lib/libtinfo.so.6
!ln -s /lib/x86_64-linux-gnu/libtinfo.so.6 /usr/local/lib/libtinfo.so.6

In [2]:
!/usr/local/envs/GPTSoVITS/bin/python -m pip uninstall -y torch torchvision torchaudio
!/usr/local/envs/GPTSoVITS/bin/python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.14.1+cu126
Uninstalling torch-2.14.1+cu126:
  Successfully uninstalled torch-2.14.1+cu126
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 173.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 328.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 219.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 43.8 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 233.7 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 203.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 189.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 180.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

## Launch WebUI


In [11]:
!cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py

Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://6e6b10ec753162c6b7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
"/usr/local/envs/GPTSoVITS/bin/python" -s GPT_SoVITS/inference_webui.py "Auto"
INFO:tools.acceleration:FlashAttention is unavailable: No module named 'flash_attn'
SoVITS_weights_v4/cheval_e2_s94_l32.pth v2 v4 True
loading sovits_v4pretrained_G <All keys matched successfully>
loading sovits_v4_lora32
Removing weight norm...
loading vocoder <All keys matched successfully>
Running on local URL:  http://0.0.0.0:9872
Running on public URL: https://1647067bb82d418fd7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
Actual Input Reference Text: 僕も僕も彼女のようになりたい。
Actual Input Target Text: 
Traceba

In [10]:
!/usr/local/envs/GPTSoVITS/bin/python -m pip install resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.3 MB/s  0:00:00


In [6]:
!zip -r /content/slicer_opt.zip /content/GPT-SoVITS/output/slicer_opt

updating: content/GPT-SoVITS/output/slicer_opt/ (stored 0%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0048759040_0048913600.wav (deflated 38%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0076675520_0076831360.wav (deflated 19%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0015748480_0015874560.wav (deflated 25%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0083774400_0083922880.wav (deflated 42%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0080939840_0081053120.wav (deflated 26%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0028034560_0028184320.wav (deflated 22%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0076297600_0076430400.wav (deflated 23%)
updating: content/GPT-SoVITS/output/slicer_opt/vocal_cheval.m4a_10.wav_0023845440_0024045440.wav (deflated 30%)
updating: content/GPT-SoVITS/output/slicer_o

In [7]:
!mkdir -p /content/GPT-SoVITS/output/cheval_sample
!unzip -o /content/GPT-SoVITS/output/cheval_sample.zip -d /content/GPT-SoVITS/output/cheval_sample

Archive:  /content/GPT-SoVITS/output/cheval_sample.zip
   creating: /content/GPT-SoVITS/output/cheval_sample/cheval/
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0015748480_0015874560.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0015897280_0016073920.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0016272960_0016472320.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0016630400_0016770560.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0016776320_0016950400.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0016953920_0017079360.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0017112000_0017254080.wav  
  inflating: /content/GPT-SoVITS/output/cheval_sample/cheval/vocal_cheval.m4a_10.wav_0017284